In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_file = "metric.csv"
trace_file = "trace.csv"
log_file = "log.csv"
error_file = "error_logs.csv"

df_metric = pd.read_csv(metric_file)
df_trace = pd.read_csv(trace_file)
df_log = pd.read_csv(log_file)
df_error = pd.read_csv(error_file)

# Parse timestamps as UTC datetimes (Unix seconds)
for df in (df_metric, df_trace, df_log, df_error):
    df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# --- metric.csv summary ---
metric_total_rows = df_metric.shape[0]
metric_unique_cmdb = pd.Series(df_metric['cmdb_id'].dropna().unique())
metric_unique_cmdb_count = metric_unique_cmdb.size
metric_unique_cmdb_sample = metric_unique_cmdb.head(20).reset_index(drop=True)

kpi_counts = (
    df_metric.groupby('kpi_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)
kpi_counts_top20 = kpi_counts.head(20)

metric_earliest = df_metric['timestamp_dt'].min()
metric_latest = df_metric['timestamp_dt'].max()

metric_overview = pd.DataFrame([{
    'total_rows': metric_total_rows,
    'unique_cmdb_count': int(metric_unique_cmdb_count),
    'unique_cmdbs_sample': metric_unique_cmdb_sample.to_list(),
    'earliest_timestamp_utc': metric_earliest,
    'latest_timestamp_utc': metric_latest
}])

# --- trace.csv summary ---
trace_total_rows = df_trace.shape[0]
trace_unique_cmdb = pd.Series(df_trace['cmdb_id'].dropna().unique())
trace_unique_cmdb_count = trace_unique_cmdb.size
trace_unique_cmdb_sample = trace_unique_cmdb.head(20).reset_index(drop=True)

trace_name_counts = (
    df_trace.groupby('trace_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)
trace_name_counts_top20 = trace_name_counts.head(20)

trace_earliest = df_trace['timestamp_dt'].min()
trace_latest = df_trace['timestamp_dt'].max()

trace_overview = pd.DataFrame([{
    'total_rows': trace_total_rows,
    'unique_cmdb_count': int(trace_unique_cmdb_count),
    'unique_cmdbs_sample': trace_unique_cmdb_sample.to_list(),
    'earliest_timestamp_utc': trace_earliest,
    'latest_timestamp_utc': trace_latest
}])

# --- log.csv summary ---
log_total_rows = df_log.shape[0]
log_unique_cmdb = pd.Series(df_log['cmdb_id'].dropna().unique())
log_unique_cmdb_count = log_unique_cmdb.size
log_unique_cmdb_sample = log_unique_cmdb.head(20).reset_index(drop=True)

log_name_counts = (
    df_log.groupby('log_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)
log_name_counts_top20 = log_name_counts.head(20)

log_earliest = df_log['timestamp_dt'].min()
log_latest = df_log['timestamp_dt'].max()

log_overview = pd.DataFrame([{
    'total_rows': log_total_rows,
    'unique_cmdb_count': int(log_unique_cmdb_count),
    'unique_cmdbs_sample': log_unique_cmdb_sample.to_list(),
    'earliest_timestamp_utc': log_earliest,
    'latest_timestamp_utc': log_latest
}])

# --- error_logs.csv summary ---
error_total_rows = df_error.shape[0]
error_unique_cmdb = pd.Series(df_error['cmdb_id'].dropna().unique())
error_unique_cmdb_count = error_unique_cmdb.size
error_unique_cmdb_sample = error_unique_cmdb.head(20).reset_index(drop=True)

error_overview = pd.DataFrame([{
    'total_rows': error_total_rows,
    'unique_cmdb_count': int(error_unique_cmdb_count),
    'unique_cmdbs_sample': error_unique_cmdb_sample.to_list()
}])

# Top 10 rows sorted by timestamp ascending (compact view)
error_top10 = (
    df_error.sort_values('timestamp_dt', ascending=True)
    .loc[:, ['timestamp_dt', 'cmdb_id', 'message']]
    .head(10)
    .reset_index(drop=True)
)

# Final compact outputs (variables shown for IPython)
metric_overview, kpi_counts_top20, trace_overview, trace_name_counts_top20, log_overview, log_name_counts_top20, error_overview, error_top10

```
Out[1]:
```
summary = (
    "Summary of telemetry files:\n\n"
    "metric.csv: 9136 rows; 68 unique cmdb_id (examples: ts-admin-basic-info-service, "
    "ts-admin-order-service, ts-auth-service, ...). Earliest timestamp 2024-01-22 15:54:00+00:00, "
    "latest 2024-01-22 16:18:00+00:00. Top KPI names (by row count, top entries): "
    "cpu 1700, mem 1700, socket 1700, diskio 1675, workload 700, latency-50 698, latency-90 698, error 265.\n\n"
    "trace.csv: 18464 rows; 28 unique cmdb_id (examples: root, ts-admin-basic-info-service, ts-admin-travel-service, ...). "
    "Earliest 2024-01-22 15:54:00+00:00, latest 2024-01-22 16:18:00+00:00. Top trace_names (top entries): "
    "trace.from_root.duration_mean 343, trace.from_root.duration_p95 343, trace.from_root.error_rate 343, "
    "trace.from_root.row_count 343, trace.from_ts-preserve-service.row_count 261, "
    "trace.from_ts-preserve-service.error_rate 261, trace.from_ts-preserve-service.duration_p95 261, "
    "trace.from_ts-preserve-service.duration_mean 261, trace.from_ts-preserve-other-service.row_count 225, "
    "trace.from_ts-preserve-other-service.error_rate 225, (more trace metrics follow up to the top 20).\n\n"
    "log.csv: 2198 rows; 47 unique cmdb_id (examples: ts-admin-basic-info-service, ts-admin-travel-service, ...). "
    "Earliest 2024-01-22 15:54:00+00:00, latest 2024-01-22 16:18:00+00:00. Top log names: log.error_count 1099, log.row_count 1099.\n\n"
    "error_logs.csv: 0 rows; no cmdb_id entries and no log messages (top 10 rows empty).\n\n"
    "Overall: metric and trace telemetry are populated with many metrics and traces across services; logs have per-minute aggregates, "
    "and there are no raw error log entries in error_logs.csv for this time window."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows  unique_cmdb_count                                unique_cmdbs_sample    earliest_timestamp_utc      latest_timestamp_utc
0        9136                 68  [ts-admin-basic-info-service, ts-admin-order-s... 2024-01-22 15:54:00+00:00 2024-01-22 16:18:00+00:00,      kpi_name  count
0         cpu   1700
1         mem   1700
2      socket   1700
3      diskio   1675
4    workload    700
5  latency-50    698
6  latency-90    698
7       error    265,    total_rows  unique_cmdb_count                                unique_cmdbs_sample    earliest_timestamp_utc      latest_timestamp_utc
0       18464                 28  [root, ts-admin-basic-info-service, ts-admin-t... 2024-01-22 15:54:00+00:00 2024-01-22 16:18:00+00:00,                                            trace_name  count
0                       trace.from_root.duration_mean    343
1                        trace.from_root.duration_p95    343
2                          trace.from_root.error_rate    343
3                           trace.from_root.row_count    343
4            trace.from_ts-preserve-service.row_count    261
5           trace.from_ts-preserve-service.error_rate    261
6         trace.from_ts-preserve-service.duration_p95    261
7        trace.from_ts-preserve-service.duration_mean    261
8      trace.from_ts-preserve-other-service.row_count    225
9     trace.from_ts-preserve-other-service.error_rate    225
10  trace.from_ts-preserve-other-service.duration_p95    225
11  trace.from_ts-preserve-other-service.duration_...    225
12                trace.to_ts-order-service.row_count    172
13               trace.to_ts-order-service.error_rate    172
14             trace.to_ts-order-service.duration_p95    172
15            trace.to_ts-order-service.duration_mean    172
16          trace.to_ts-station-service.duration_mean    169
17           trace.to_ts-station-service.duration_p95    169
18             trace.to_ts-station-service.error_rate    169
19              trace.to_ts-station-service.row_count    169,    total_rows  unique_cmdb_count                                unique_cmdbs_sample    earliest_timestamp_utc      latest_timestamp_utc
0        2198                 47  [ts-admin-basic-info-service, ts-admin-travel-... 2024-01-22 15:54:00+00:00 2024-01-22 16:18:00+00:00,           log_name  count
0  log.error_count   1099
1    log.row_count   1099,    total_rows  unique_cmdb_count unique_cmdbs_sample
0           0                  0                  [], Empty DataFrame
Columns: [timestamp_dt, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse df_metric loaded previously. If not present, load it.
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure timestamp_dt exists and value is numeric
if 'timestamp_dt' not in df_metric.columns:
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Candidate components
candidates = [
    "ts-auth-service",
    "ts-order-service",
    "ts-route-service",
    "ts-train-service",
    "ts-travel-service"
]

# Filter metric rows to only candidate components
df_candidates = df_metric[df_metric['cmdb_id'].isin(candidates)].copy()

# Incident window (UTC)
start_window = pd.to_datetime("2024-01-22 15:51:51", utc=True)
end_window = pd.to_datetime("2024-01-22 16:21:51", utc=True)

# Compute rows_total, earliest, latest, global p95, and rows_in_window per (cmdb_id, kpi_name)
grouped = df_candidates.groupby(['cmdb_id', 'kpi_name'], dropna=False)

agg_df = grouped.agg(
    rows_total = ('value', 'size'),
    p95_value = ('value', lambda s: s.quantile(0.95) if s.notna().any() else float('nan')),
    earliest_timestamp_utc = ('timestamp_dt', 'min'),
    latest_timestamp_utc = ('timestamp_dt', 'max'),
).reset_index()

# Compute rows_in_window per group
in_window_mask = (df_candidates['timestamp_dt'] >= start_window) & (df_candidates['timestamp_dt'] <= end_window)
rows_in_window = (
    df_candidates.loc[in_window_mask]
    .groupby(['cmdb_id', 'kpi_name'])
    .size()
    .rename('rows_in_window')
    .reset_index()
)

# Merge rows_in_window (missing groups -> 0)
result = pd.merge(agg_df, rows_in_window, on=['cmdb_id', 'kpi_name'], how='left')
result['rows_in_window'] = result['rows_in_window'].fillna(0).astype(int)

# Order and sort
result = result[[
    'cmdb_id',
    'kpi_name',
    'rows_total',
    'p95_value',
    'earliest_timestamp_utc',
    'latest_timestamp_utc',
    'rows_in_window'
]].sort_values(['rows_total', 'cmdb_id', 'kpi_name'], ascending=[False, True, True])

# Limit to at most 100 rows (compact)
result_table = result.head(100)

result_table

```
Out[2]:
```
Summary of metric.csv for the specified candidate components (ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service):

- Found 38 distinct (cmdb_id, kpi_name) groups for the five candidate services.
- Each group has rows_total = 25 and rows_in_window = 25 — i.e., all samples for these groups fall inside the incident window.
- Time range for these groups: earliest = 2024-01-22 15:54:00 UTC, latest = 2024-01-22 16:18:00 UTC.
- Global P95 'value' varies by KPI and service. Example P95 values (per group):
  - ts-auth-service:
    - cpu p95 ≈ 15.06
    - mem p95 ≈ 243,084,000
    - diskio p95 ≈ 63,309
    - latency-50 p95 ≈ 0.263
    - latency-90 p95 ≈ 0.841
  - ts-travel-service:
    - mem p95 ≈ 272,582,400
    - socket p95 ≈ 23.43
    - latency-50 p95 ≈ 0.315
    - latency-90 p95 ≈ 2.344
    - workload p95 ≈ 10.27
- All candidate groups have full coverage within the incident window (no missing per-group samples in-window). P95 magnitudes indicate expected differences by KPI (e.g., memory values in 1e8 range, disk I/O large, latencies sub-second to a few seconds).

If you want the full per-group table (the 38 rows) displayed, I can show it.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  rows_total     p95_value    earliest_timestamp_utc      latest_timestamp_utc  rows_in_window
0     ts-auth-service         cpu          25  1.506390e+01 2024-01-22 15:54:00+00:00 2024-01-22 16:18:00+00:00              25
1     ts-auth-service      diskio          25  6.330939e+04 2024-01-22 15:54:00+00:00 2024-01-22 16:18:00+00:00              25
2     ts-auth-service  latency-50          25  2.631019e-01 2024-01-22 15:54:00+00:00 2024-01-22 16:18:00+00:00              25
3     ts-auth-service  latency-90          25  8.410099e-01 2024-01-22 15:54:00+00:00 2024-01-22 16:18:00+00:00              25
4     ts-auth-service         mem          25  2.430840e+08 2024-01-22 15:54:00+00:00 2024-01-22 16:18:00+00:00              25
..                ...         ...         ...           ...                       ...                       ...             ...
33  ts-travel-service  latency-50          25  3.152417e-01 2024-01-22 15:54:00+00:00 2024-01-22 16:18:00+00:00              25
34  ts-travel-service  latency-90          25  2.344489e+00 2024-01-22 15:54:00+00:00 2024-01-22 16:18:00+00:00              25
35  ts-travel-service         mem          25  2.725824e+08 2024-01-22 15:54:00+00:00 2024-01-22 16:18:00+00:00              25
36  ts-travel-service      socket          25  2.342718e+01 2024-01-22 15:54:00+00:00 2024-01-22 16:18:00+00:00              25
37  ts-travel-service    workload          25  1.027184e+01 2024-01-22 15:54:00+00:00 2024-01-22 16:18:00+00:00              25

[38 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timestamp_dt and numeric value
if 'timestamp_dt' not in df_metric.columns:
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Candidate components
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Restrict to candidate components
df_candidates_all = df_metric[df_metric['cmdb_id'].isin(candidates)].copy()

# Compute global P95 per (cmdb_id, kpi_name) using entire available data for each group
global_p95_df = (
    df_candidates_all
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value':'global_p95'})
)

# Incident window (UTC)
start_window = pd.to_datetime("2024-01-22 15:51:51", utc=True)
end_window   = pd.to_datetime("2024-01-22 16:21:51", utc=True)

# Merge global_p95 into the full candidate series
merged = pd.merge(df_candidates_all, global_p95_df, on=['cmdb_id','kpi_name'], how='left')

# Filter to window using a mask aligned with merged
mask_window = (merged['timestamp_dt'] >= start_window) & (merged['timestamp_dt'] <= end_window)
window_df = merged.loc[mask_window].copy()

# rows_in_window and max_value_in_window per group
rows_in_window_df = (
    window_df
    .groupby(['cmdb_id','kpi_name'], dropna=False)
    .agg(rows_in_window=('value','size'),
         max_value_in_window=('value','max'))
    .reset_index()
)

# anomaly rows where value >= global_p95 (global_p95 computed before filtering)
anoms_df = window_df[window_df['value'] >= window_df['global_p95']].copy()

anoms_agg = (
    anoms_df
    .groupby(['cmdb_id','kpi_name'], dropna=False)
    .agg(anomaly_count=('value','size'),
         earliest_anomaly_timestamp_utc=('timestamp_dt','min'))
    .reset_index()
)

# Combine aggregates with global_p95
result = pd.merge(rows_in_window_df, global_p95_df, on=['cmdb_id','kpi_name'], how='left')
result = pd.merge(result, anoms_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill missing anomaly_count with 0 and keep earliest_anomaly_timestamp_utc as NaT when no anomalies
result['anomaly_count'] = result['anomaly_count'].fillna(0).astype(int)

# Compute ratio_max_p95, handle division by zero
result['ratio_max_p95'] = result['max_value_in_window'] / result['global_p95']
result.loc[result['global_p95'] == 0, 'ratio_max_p95'] = np.nan

# Filter to groups with anomaly_count > 0
result_pos = result[result['anomaly_count'] > 0].copy()

# Select and order columns as requested
result_pos = result_pos[[
    'cmdb_id',
    'kpi_name',
    'rows_in_window',
    'anomaly_count',
    'earliest_anomaly_timestamp_utc',
    'max_value_in_window',
    'global_p95',
    'ratio_max_p95'
]]

# Sort by anomaly_count desc, then ratio_max_p95 desc, limit to top 20
result_pos = result_pos.sort_values(['anomaly_count', 'ratio_max_p95'], ascending=[False, False]).head(20)

# Compact display
result_pos

```
Out[3]:
```
Summary of findings (candidate services only):

- Total groups with anomalies (value >= group global P95) returned: 20 groups (top 20 shown).
- Three groups show anomaly_count = 25 for the "error" KPI:
  - ts-route-service, ts-train-service, ts-travel-service — these have global_p95 = 0 and max_value_in_window = 0, so every sample meets value >= global_p95. The ratio is undefined (NaN). These are not informative anomalies (baseline P95 is zero).
- The most notable true anomalies (global_p95 > 0) by severity (ratio_max_p95) are:
  1. ts-order-service — latency-90: anomaly_count = 2, earliest anomaly 2024-01-22 16:08:00 UTC, max_in_window = 0.109898, global_p95 = 0.023696, ratio ≈ 4.64 (largest relative spike).
  2. ts-auth-service — diskio: anomaly_count = 2, earliest 2024-01-22 15:54:00 UTC, max = 75213.47, global_p95 = 63309.39, ratio ≈ 1.19.
  3. ts-auth-service — latency-50: anomaly_count = 2, earliest 2024-01-22 15:55:00 UTC, max = 0.309233, global_p95 = 0.263102, ratio ≈ 1.18.
  4. ts-travel-service — socket: anomaly_count = 2, earliest 2024-01-22 16:13:00 UTC, max = 27.10, global_p95 = 23.427179, ratio ≈ 1.16.
  5. ts-route-service — diskio: anomaly_count = 2, earliest 2024-01-22 15:54:00 UTC, max = 157224.91, global_p95 = 136782.61, ratio ≈ 1.15.
- Most other flagged groups have anomaly_count = 2 and modest ratio_max_p95 (≈1.02–1.12), e.g., cpu, latency-90, workload, socket, diskio across the candidate services.
- All anomalies occurred inside the incident window; rows_in_window for these groups is 25 (full coverage).

Interpretation: aside from the three "error" KPIs (which are zero-based and trigger trivial matches), the strongest relative spike is ts-order-service latency-90 (ratio ~4.6 at ~16:08 UTC). Several services show modest increases in disk I/O, CPU, latency, socket and workload (ratio ≈1.03–1.19) with anomaly_count = 2 samples each.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  rows_in_window  anomaly_count earliest_anomaly_timestamp_utc  max_value_in_window     global_p95  ratio_max_p95
16   ts-route-service       error              25             25      2024-01-22 15:54:00+00:00             0.000000       0.000000            NaN
24   ts-train-service       error              25             25      2024-01-22 15:54:00+00:00             0.000000       0.000000            NaN
32  ts-travel-service       error              25             25      2024-01-22 15:54:00+00:00             0.000000       0.000000            NaN
10   ts-order-service  latency-90              25              2      2024-01-22 16:08:00+00:00             0.109898       0.023696       4.637842
1     ts-auth-service      diskio              25              2      2024-01-22 15:54:00+00:00         75213.467054   63309.392346       1.188030
2     ts-auth-service  latency-50              25              2      2024-01-22 15:55:00+00:00             0.309233       0.263102       1.175336
36  ts-travel-service      socket              25              2      2024-01-22 16:13:00+00:00            27.100000      23.427179       1.156776
15   ts-route-service      diskio              25              2      2024-01-22 15:54:00+00:00        157224.912944  136782.614712       1.149451
3     ts-auth-service  latency-90              25              2      2024-01-22 15:55:00+00:00             0.941672       0.841010       1.119692
14   ts-route-service         cpu              25              2      2024-01-22 15:55:00+00:00             5.302863       4.737774       1.119273
0     ts-auth-service         cpu              25              2      2024-01-22 15:54:00+00:00            16.523953      15.063903       1.096924
9    ts-order-service  latency-50              25              2      2024-01-22 16:08:00+00:00             0.011166       0.010274       1.086816
20   ts-route-service      socket              25              2      2024-01-22 16:14:00+00:00            26.850000      24.974359       1.075103
22   ts-train-service         cpu              25              2      2024-01-22 15:59:00+00:00             3.750651       3.608809       1.039304
23   ts-train-service      diskio              25              2      2024-01-22 15:56:00+00:00         65378.872180   63321.515734       1.032491
13   ts-order-service    workload              25              2      2024-01-22 15:56:00+00:00             7.438867       7.221570       1.030090
34  ts-travel-service  latency-90              25              2      2024-01-22 16:16:00+00:00             2.413471       2.344489       1.029423
29   ts-train-service    workload              25              2      2024-01-22 15:56:00+00:00            22.594533      21.951243       1.029305
37  ts-travel-service    workload              25              2      2024-01-22 15:55:00+00:00            10.547900      10.271840       1.026875
30  ts-travel-service         cpu              25              2      2024-01-22 15:56:00+00:00             3.805925       3.718273       1.023573```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import re

# Reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure timestamp parsed and value numeric
if 'timestamp_dt' not in df_trace.columns:
    df_trace['timestamp_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# 1) Compute global P95 per trace_name using the entire series
global_p95_df = (
    df_trace
    .groupby('trace_name', dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# Incident window (UTC)
start_window = pd.to_datetime("2024-01-22 15:51:51", utc=True)
end_window   = pd.to_datetime("2024-01-22 16:21:51", utc=True)

# 2) Filter trace rows to the incident window
window_mask = (df_trace['timestamp_dt'] >= start_window) & (df_trace['timestamp_dt'] <= end_window)
window_df = df_trace.loc[window_mask].copy()

# Merge global_p95 into windowed rows for anomaly detection
window_merged = window_df.merge(global_p95_df, on='trace_name', how='left')

# Compute rows_in_window and max_value_in_window per trace_name
rows_in_window_df = (
    window_merged
    .groupby('trace_name', dropna=False)
    .agg(rows_in_window=('value','size'),
         max_value_in_window=('value','max'))
    .reset_index()
)

# Compute anomalies where value >= global_p95
anoms_df = window_merged[window_merged['value'] >= window_merged['global_p95']].copy()
anoms_agg = (
    anoms_df
    .groupby('trace_name', dropna=False)
    .agg(anomaly_count=('value','size'),
         earliest_anomaly_timestamp_utc=('timestamp_dt','min'))
    .reset_index()
)

# Combine aggregates
result = global_p95_df.merge(rows_in_window_df, on='trace_name', how='left')
result = result.merge(anoms_agg, on='trace_name', how='left')

# Fill missing numeric aggregates
result['rows_in_window'] = result['rows_in_window'].fillna(0).astype(int)
result['max_value_in_window'] = result['max_value_in_window'].fillna(np.nan)
result['anomaly_count'] = result['anomaly_count'].fillna(0).astype(int)
result['earliest_anomaly_timestamp_utc'] = pd.to_datetime(result['earliest_anomaly_timestamp_utc'])

# Compute ratio_max_p95, handle division by zero
result['ratio_max_p95'] = result['max_value_in_window'] / result['global_p95']
result.loc[result['global_p95'] == 0, 'ratio_max_p95'] = np.nan

# Filter trace_name entries that contain any of the specified substrings
substrings = [
    'order','train','route','travel','auth',
    'from_root','from_ts-preserve',
    'to_ts-order-service','to_ts-train-service','to_ts-route-service','to_ts-travel-service','to_ts-auth-service'
]
# build regex pattern, escape substrings to be safe
pattern = '|'.join(re.escape(s) for s in substrings)
mask_select = result['trace_name'].str.contains(pattern, regex=True, na=False)

filtered = result.loc[mask_select].copy()

# Select and order columns as requested
filtered = filtered[[
    'trace_name',
    'rows_in_window',
    'anomaly_count',
    'earliest_anomaly_timestamp_utc',
    'max_value_in_window',
    'global_p95',
    'ratio_max_p95'
]]

# Sort by ratio_max_p95 desc then anomaly_count desc and limit to top 20
filtered_sorted = filtered.sort_values(['ratio_max_p95','anomaly_count'], ascending=[False, False]).head(20)

# Compact result for display
filtered_sorted

```
Out[4]:
```
Summary of trace anomalies within the incident window (top findings):

- Strongest relative spike:
  - trace.from_ts-preserve-service.duration_mean — ratio ≈ 4.24, anomaly_count = 14, rows_in_window = 261, earliest anomaly 2024-01-22 16:07:00 UTC, max = 1.9318 (global P95 = 0.4558).

- Other high-severity trace signals:
  - trace.from_ts-preserve-other-service.duration_* (duration_p95/mean) — ratio ≈ 3.07, anomaly_count = 12, rows_in_window = 225, earliest 2024-01-22 15:56:00 UTC, max ≈ 1.9469 (global P95 ≈ 0.6348).
  - trace.from_ts-preserve-other-service.row_count — ratio ≈ 2.53, anomaly_count = 13, rows_in_window = 225, earliest 2024-01-22 15:56:00 UTC, max = 48 (global P95 = 19).
  - trace.from_ts-preserve-service.row_count — ratio ≈ 1.84, anomaly_count = 14, rows_in_window = 261, earliest 2024-01-22 15:55:00 UTC, max = 70 (global P95 = 38).

- Order-related traces show notable anomalies:
  - trace.to_ts-order-other-service.duration_p95 — ratio ≈ 1.81, anomaly_count = 9, rows_in_window = 166, earliest 2024-01-22 16:03:00 UTC.
  - trace.from_ts-order-other-service.duration_p95 and duration_mean — ratios ≈ 1.73 and 1.26, anomaly_count = 3, rows_in_window = 48.
  - trace.to_ts-order-service.duration_p95 and row_count — ratio ≈ 1.48 and 1.38, anomaly_count = 9, rows_in_window = 172.

- Auth / root traces also flagged:
  - trace.from_ts-auth-service.duration_p95 and duration_mean — small counts (2) but elevated ratios ≈ 1.61 and 1.24, earliest anomalies at ~15:54–16:00 UTC.
  - trace.from_root.duration_p95 and duration_mean — anomaly_count = 18, rows_in_window = 343, ratios ≈ 1.56 and 1.41, earliest anomalies ~16:07:00 UTC.

- Travel/train route row_count traces show modest spikes:
  - trace.from_ts-travel-service.row_count and trace.to_ts-travel-service.row_count — ratios ≈ 1.25, anomaly_count = 8, rows_in_window ≈ 149–150, earliest ~15:55 UTC.
  - Several train/order/route related duration or row_count metrics have smaller but consistent anomaly counts (multiple groups with anomaly_count > 0).

Interpretation:
- The highest relative anomalies are concentrated on traces from the "ts-preserve" flows (duration and row_count) and several order-related downstream traces; many anomalies cluster around ~16:03–16:07 UTC.
- from_root traces also show broad anomalies (many rows flagged).
- These trace-level spikes point to degraded latencies/latency-percentiles and increased row counts in preserve/order-related flows during the incident window; they are top candidates for investigation.

The original code execution output of IPython Kernel is also provided below for reference:

trace_name  rows_in_window  anomaly_count earliest_anomaly_timestamp_utc  max_value_in_window   global_p95  ratio_max_p95
68        trace.from_ts-preserve-service.duration_mean             261             14      2024-01-22 16:07:00+00:00             1.931765     0.455783       4.238347
64   trace.from_ts-preserve-other-service.duration_...             225             12      2024-01-22 15:56:00+00:00             1.946923     0.634761       3.067174
67      trace.from_ts-preserve-other-service.row_count             225             13      2024-01-22 15:56:00+00:00            48.000000    19.000000       2.526316
71            trace.from_ts-preserve-service.row_count             261             14      2024-01-22 15:55:00+00:00            70.000000    38.000000       1.842105
161       trace.to_ts-order-other-service.duration_p95             166              9      2024-01-22 16:03:00+00:00             0.777832     0.430780       1.805637
53      trace.from_ts-order-other-service.duration_p95              48              3      2024-01-22 16:03:00+00:00             0.762093     0.439798       1.732827
17             trace.from_ts-auth-service.duration_p95              25              2      2024-01-22 15:54:00+00:00             0.006885     0.004263       1.614935
1                         trace.from_root.duration_p95             343             18      2024-01-22 16:07:00+00:00             6.798954     4.357334       1.560347
165             trace.to_ts-order-service.duration_p95             172              9      2024-01-22 16:03:00+00:00             0.849479     0.572058       1.484953
0                        trace.from_root.duration_mean             343             18      2024-01-22 16:07:00+00:00             5.925311     4.202971       1.409791
124             trace.to_ts-auth-service.duration_mean              50              3      2024-01-22 15:55:00+00:00             0.451481     0.326161       1.384227
167                trace.to_ts-order-service.row_count             172              9      2024-01-22 15:55:00+00:00           589.000000   427.600000       1.377456
57            trace.from_ts-order-service.duration_p95              50              3      2024-01-22 16:03:00+00:00             0.830222     0.614656       1.350711
160      trace.to_ts-order-other-service.duration_mean             166              9      2024-01-22 16:03:00+00:00             0.545873     0.413535       1.320015
52     trace.from_ts-order-other-service.duration_mean              48              3      2024-01-22 16:03:00+00:00             0.529025     0.419979       1.259645
163          trace.to_ts-order-other-service.row_count             166              9      2024-01-22 15:55:00+00:00           773.000000   614.500000       1.257933
103             trace.from_ts-travel-service.row_count             150              8      2024-01-22 15:55:00+00:00          1865.000000  1488.950000       1.252561
211               trace.to_ts-travel-service.row_count             149              8      2024-01-22 15:55:00+00:00          1865.000000  1490.400000       1.251342
16            trace.from_ts-auth-service.duration_mean              25              2      2024-01-22 16:00:00+00:00             0.004624     0.003714       1.244859
105         trace.from_ts-travel2-service.duration_p95             150              8      2024-01-22 16:07:00+00:00             0.564106     0.460591       1.224744```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse df_log if present, otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure timestamp_dt parsed and value numeric
if 'timestamp_dt' not in df_log.columns:
    df_log['timestamp_dt'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Candidate components
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Restrict to candidate components for threshold computation and window analysis
df_logs_cand = df_log[df_log['cmdb_id'].isin(candidates)].copy()

# 1) Compute global P95 per (cmdb_id, log_name) using the full available data for those groups
global_p95_logs = (
    df_logs_cand
    .groupby(['cmdb_id','log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value':'global_p95'})
)

# Incident window (UTC)
start_window = pd.to_datetime("2024-01-22 15:51:51", utc=True)
end_window   = pd.to_datetime("2024-01-22 16:21:51", utc=True)

# 2) Filter rows to the incident window
mask_window = (df_logs_cand['timestamp_dt'] >= start_window) & (df_logs_cand['timestamp_dt'] <= end_window)
window_logs = df_logs_cand.loc[mask_window].copy()

# Merge global_p95 into window rows for anomaly detection
window_merged = window_logs.merge(global_p95_logs, on=['cmdb_id','log_name'], how='left')

# rows_in_window and max_value_in_window per group
rows_in_window_df = (
    window_merged
    .groupby(['cmdb_id','log_name'], dropna=False)
    .agg(rows_in_window=('value','size'),
         max_value_in_window=('value','max'))
    .reset_index()
)

# anomaly rows where value >= global_p95 (global threshold computed before filtering)
anoms_df = window_merged[window_merged['value'] >= window_merged['global_p95']].copy()
anoms_agg = (
    anoms_df
    .groupby(['cmdb_id','log_name'], dropna=False)
    .agg(anomaly_count=('value','size'),
         earliest_anomaly_timestamp_utc=('timestamp_dt','min'))
    .reset_index()
)

# Combine aggregates with global_p95 (keep only groups seen in candidates)
result = global_p95_logs.merge(rows_in_window_df, on=['cmdb_id','log_name'], how='left')
result = result.merge(anoms_agg, on=['cmdb_id','log_name'], how='left')

# Fill missing values
result['rows_in_window'] = result['rows_in_window'].fillna(0).astype(int)
result['max_value_in_window'] = result['max_value_in_window'].fillna(np.nan)
result['anomaly_count'] = result['anomaly_count'].fillna(0).astype(int)
result['earliest_anomaly_timestamp_utc'] = pd.to_datetime(result['earliest_anomaly_timestamp_utc'])

# Compute ratio_max_p95, handle division by zero
result['ratio_max_p95'] = result['max_value_in_window'] / result['global_p95']
result.loc[result['global_p95'] == 0, 'ratio_max_p95'] = np.nan

# Keep only groups with anomaly_count > 0
result_pos = result[result['anomaly_count'] > 0].copy()

# Select and order columns as requested
result_pos = result_pos[[
    'cmdb_id',
    'log_name',
    'rows_in_window',
    'anomaly_count',
    'earliest_anomaly_timestamp_utc',
    'max_value_in_window',
    'global_p95',
    'ratio_max_p95'
]]

# Sort by ratio_max_p95 desc then anomaly_count desc, limit to top 20
result_pos = result_pos.sort_values(['ratio_max_p95','anomaly_count'], ascending=[False, False]).head(20)

# Compact display
result_pos

```
Out[5]:
```
Summary of log.csv anomalies for the five candidate services:

- Useful non-trivial anomalies (log.row_count):
  - ts-travel-service — rows_in_window=25, anomaly_count=2, earliest anomaly 2024-01-22 15:55:00 UTC, max_in_window=2094, global_p95=1981.0, ratio ≈ 1.057 (largest relative spike among logs).
  - ts-order-service  — rows_in_window=25, anomaly_count=2, earliest anomaly 2024-01-22 15:55:00 UTC, max_in_window=493,  global_p95=477.0, ratio ≈ 1.034.
  - ts-auth-service   — rows_in_window=25, anomaly_count=2, earliest anomaly 2024-01-22 16:02:00 UTC, max_in_window=362,  global_p95=356.8, ratio ≈ 1.015.
  - ts-route-service  — rows_in_window=25, anomaly_count=2, earliest anomaly 2024-01-22 16:00:00 UTC, max_in_window=1071, global_p95=1063.8, ratio ≈ 1.007.

- log.error_count for ts-auth, ts-order, ts-route, ts-travel:
  - Each shows anomaly_count=25 with max_in_window=0 and global_p95=0 (ratio undefined). This is a trivial match because the P95 baseline is zero — not an informative anomaly.

Interpretation:
- Small but measurable spikes in log.row_count occurred for travel and order (most significant), and minor spikes for auth and route. The log.error_count results are not meaningful (P95==0), so focus investigation on the row_count increases (especially ts-travel-service at ~15:55 UTC).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  rows_in_window  anomaly_count earliest_anomaly_timestamp_utc  max_value_in_window  global_p95  ratio_max_p95
7  ts-travel-service    log.row_count              25              2      2024-01-22 15:55:00+00:00                 2094      1981.0       1.057042
3   ts-order-service    log.row_count              25              2      2024-01-22 15:55:00+00:00                  493       477.0       1.033543
1    ts-auth-service    log.row_count              25              2      2024-01-22 16:02:00+00:00                  362       356.8       1.014574
5   ts-route-service    log.row_count              25              2      2024-01-22 16:00:00+00:00                 1071      1063.8       1.006768
0    ts-auth-service  log.error_count              25             25      2024-01-22 15:54:00+00:00                    0         0.0            NaN
2   ts-order-service  log.error_count              25             25      2024-01-22 15:54:00+00:00                    0         0.0            NaN
4   ts-route-service  log.error_count              25             25      2024-01-22 15:54:00+00:00                    0         0.0            NaN
6  ts-travel-service  log.error_count              25             25      2024-01-22 15:54:00+00:00                    0         0.0            NaN```
```